In [2]:
# Fabric Notebook Source — Gold Layer Pipeline
# Workspace : Daily_Sales_Analytics
# Lakehouse  : Pulse_Sales_Gold  (attach as default lakehouse)
# Trigger    : Automatically after Silver notebook completes successfully
#
# ─────────────────────────────────────────────────────────────────────────────
# HOW TO IMPORT INTO FABRIC
# 1. Open your Fabric workspace → New → Import notebook → upload this file
# 2. Create a new lakehouse named Pulse_Sales_Gold and attach as default
# 3. Also add Pulse_Sales_Silver as a secondary lakehouse (read source)
# ─────────────────────────────────────────────────────────────────────────────


# ── Notebook Cell 1 ──────────────────────────────────────────────────────────
# TITLE: Gold Layer — Sales Pipeline
# Medallion layer : Gold
# Source          : Pulse_Sales_Silver → Tables/sales_silver (Delta)
# Destination     : Pulse_Sales_Gold   → 4 aggregated Delta tables
# Supporting tbls : gold_audit_log



StatementMeta(, bea94974-b6f7-41e7-9c6b-ebf55260c66c, 4, Finished, Available, Finished, False)

In [3]:

# ── Notebook Cell 2 ──────────────────────────────────────────────────────────
# TITLE: Imports & configuration

import uuid
import time
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType,
    DecimalType, TimestampType, FloatType
)
from delta.tables import DeltaTable

# ── Path & name constants ─────────────────────────────────────────────────────

SILVER_LAKEHOUSE = "Pulse_Sales_Silver"
GOLD_LAKEHOUSE   = "Pulse_Sales_Gold"

# Source
SILVER_TABLE = f"{SILVER_LAKEHOUSE}.sales_silver"

# Gold output tables
GOLD_DAILY_STORE_PATH    = "Tables/gold_daily_store_revenue"
GOLD_DAILY_PRODUCT_PATH  = "Tables/gold_daily_product_revenue"
GOLD_MONTHLY_PATH        = "Tables/gold_monthly_summary"
GOLD_STORE_PRODUCT_PATH  = "Tables/gold_store_product_performance"
GOLD_AUDIT_PATH          = "Tables/gold_audit_log"

GOLD_DAILY_STORE_NAME    = "gold_daily_store_revenue"
GOLD_DAILY_PRODUCT_NAME  = "gold_daily_product_revenue"
GOLD_MONTHLY_NAME        = "gold_monthly_summary"
GOLD_STORE_PRODUCT_NAME  = "gold_store_product_performance"
GOLD_AUDIT_NAME          = "gold_audit_log"

# Override — write without lakehouse prefix since Gold is the default lakehouse
# Fabric resolves saveAsTable names relative to the default lakehouse
GOLD_LAKEHOUSE = "Pulse_Sales_Gold"

print("Gold layer configuration loaded.")
print(f"  Source  : {SILVER_TABLE}")
print(f"  Target  : {GOLD_LAKEHOUSE}")



StatementMeta(, bea94974-b6f7-41e7-9c6b-ebf55260c66c, 5, Finished, Available, Finished, False)

Gold layer configuration loaded.
  Source  : Pulse_Sales_Silver.sales_silver
  Target  : Pulse_Sales_Gold


In [4]:

# ── Notebook Cell 3 ──────────────────────────────────────────────────────────
# TITLE: Audit log schema & helper

AUDIT_SCHEMA = StructType([
    StructField("run_id",            StringType(),    nullable=False),
    StructField("table_name",        StringType(),    nullable=False),
    StructField("processed_at",      TimestampType(), nullable=False),
    StructField("rows_written",      LongType(),      nullable=True),
    StructField("gold_status",       StringType(),    nullable=False),
    StructField("duration_seconds",  FloatType(),     nullable=True),
    StructField("error_message",     StringType(),    nullable=True),
])

# Shared run ID — all 4 tables in one execution share the same run_id
RUN_ID    = str(uuid.uuid4())
RUN_START = time.time()

print(f"Run ID : {RUN_ID}")


def write_audit(table_name, rows_written, status, duration, error_message=None):
    """Write a single row to gold_audit_log."""
    audit_row = spark.createDataFrame(
        [{
            "run_id":           RUN_ID,
            "table_name":       table_name,
            "processed_at":     datetime.now(timezone.utc),
            "rows_written":     rows_written,
            "gold_status":      status,
            "duration_seconds": duration,
            "error_message":    error_message,
        }],
        schema=AUDIT_SCHEMA
    )
    audit_row.write.format("delta").mode("append").saveAsTable(
        GOLD_AUDIT_NAME
    )

def merge_gold_table(df, table_path, table_name, merge_keys: list):
    if not DeltaTable.isDeltaTable(spark, table_path):
        # Let Fabric manage the path — do not pass option("path")
        (df.write
            .format("delta")
            .partitionBy("Year", "Month")
            .mode("overwrite")
            .saveAsTable(table_name))
        print(f"  Created {table_name}")
    else:
        # Use table name to resolve — not file path
        delta_table = DeltaTable.forName(spark, table_name)
        merge_condition = " AND ".join(
            [f"target.{k} = source.{k}" for k in merge_keys]
        )
        (delta_table.alias("target")
            .merge(df.alias("source"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print(f"  Merged into {table_name}")


print("Helpers defined.")



StatementMeta(, bea94974-b6f7-41e7-9c6b-ebf55260c66c, 6, Finished, Available, Finished, False)

Run ID : 2714fff8-eafc-474b-85a8-d05fbda7919d
Helpers defined.


In [5]:

# ── Notebook Cell 4 ──────────────────────────────────────────────────────────
# TITLE: Read silver source

df_silver = spark.table(SILVER_TABLE)
total_silver_rows = df_silver.count()

print(f"Silver rows available : {total_silver_rows}")

if total_silver_rows == 0:
    print("No data in silver — nothing to aggregate. Exiting.")
    mssparkutils.notebook.exit("NO_SILVER_DATA")

# Cache silver — it is read 4 times below
df_silver.cache()
print("Silver data cached.")



StatementMeta(, bea94974-b6f7-41e7-9c6b-ebf55260c66c, 7, Finished, Available, Finished, False)

Silver rows available : 14
Silver data cached.


In [6]:

# ── Notebook Cell 5 ──────────────────────────────────────────────────────────
# TITLE: Gold Table 1 — gold_daily_store_revenue
#
# One row per Date + StoreID combination.
# Answers: how did each store perform each day?
# Power BI use: daily store trend line, store comparison bar chart.

t_start = time.time()

try:
    df_daily_store = (df_silver
        .groupBy("Date", "StoreID", "Year", "Month", "Day")
        .agg(
            F.round(F.sum("TotalRevenue"),    2).alias("TotalRevenue"),
            F.count("TransactionID")           .alias("TransactionCount"),
            F.sum("Quantity")                  .alias("TotalUnitsSold"),
            F.round(F.avg("TotalRevenue"),    2).alias("AvgOrderValue"),
            F.round(F.max("TotalRevenue"),    2).alias("MaxOrderValue"),
            F.round(F.min("TotalRevenue"),    2).alias("MinOrderValue"),
        )
        .withColumn("_gold_loaded_at", F.current_timestamp())
        .withColumn("_gold_run_id",    F.lit(RUN_ID))
        .orderBy("Date", "StoreID")
    )

    rows = df_daily_store.count()

    merge_gold_table(
        df         = df_daily_store,
        table_path = GOLD_DAILY_STORE_PATH,
        table_name = GOLD_DAILY_STORE_NAME,
        merge_keys = ["Date", "StoreID"]
    )

    duration = round(time.time() - t_start, 2)
    write_audit(GOLD_DAILY_STORE_NAME, rows, "SUCCESS", duration)
    print(f"   {GOLD_DAILY_STORE_NAME} — {rows} rows in {duration}s")

except Exception as e:
    duration = round(time.time() - t_start, 2)
    write_audit(GOLD_DAILY_STORE_NAME, 0, "FAILED", duration, str(e))
    print(f"   {GOLD_DAILY_STORE_NAME} FAILED: {e}")


StatementMeta(, bea94974-b6f7-41e7-9c6b-ebf55260c66c, 8, Finished, Available, Finished, False)

  Created gold_daily_store_revenue
   gold_daily_store_revenue — 9 rows in 13.31s


In [7]:

# ── Notebook Cell 6 ──────────────────────────────────────────────────────────
# TITLE: Gold Table 2 — gold_daily_product_revenue
#
# One row per Date + ProductID combination.
# Answers: which products sold best each day?
# Power BI use: product ranking table, daily product trend chart.

t_start = time.time()

try:
    df_daily_product = (df_silver
        .groupBy("Date", "ProductID", "Year", "Month", "Day")
        .agg(
            F.round(F.sum("TotalRevenue"),    2).alias("TotalRevenue"),
            F.sum("Quantity")                  .alias("UnitsSold"),
            F.count("TransactionID")           .alias("TransactionCount"),
            F.round(F.avg("UnitPrice"),        2).alias("AvgUnitPrice"),
            F.round(F.max("UnitPrice"),        2).alias("MaxUnitPrice"),
            F.round(F.min("UnitPrice"),        2).alias("MinUnitPrice"),
        )
        .withColumn("_gold_loaded_at", F.current_timestamp())
        .withColumn("_gold_run_id",    F.lit(RUN_ID))
        .orderBy("Date", "ProductID")
    )

    rows = df_daily_product.count()

    merge_gold_table(
        df         = df_daily_product,
        table_path = GOLD_DAILY_PRODUCT_PATH,
        table_name = GOLD_DAILY_PRODUCT_NAME,
        merge_keys = ["Date", "ProductID"]
    )

    duration = round(time.time() - t_start, 2)
    write_audit(GOLD_DAILY_PRODUCT_NAME, rows, "SUCCESS", duration)
    print(f"   {GOLD_DAILY_PRODUCT_NAME} — {rows} rows in {duration}s")

except Exception as e:
    duration = round(time.time() - t_start, 2)
    write_audit(GOLD_DAILY_PRODUCT_NAME, 0, "FAILED", duration, str(e))
    print(f"   {GOLD_DAILY_PRODUCT_NAME} FAILED: {e}")



StatementMeta(, bea94974-b6f7-41e7-9c6b-ebf55260c66c, 9, Finished, Available, Finished, False)

  Created gold_daily_product_revenue
   gold_daily_product_revenue — 9 rows in 7.35s


In [8]:


# ── Notebook Cell 7 ──────────────────────────────────────────────────────────
# TITLE: Gold Table 3 — gold_monthly_summary
#
# One row per Year + Month.
# Answers: what was the overall business performance each month?
# Power BI use: executive KPI cards, MoM trend line, summary table.

t_start = time.time()

try:
    df_monthly = (df_silver
        .groupBy("Year", "Month")
        .agg(
            F.round(F.sum("TotalRevenue"),    2).alias("TotalRevenue"),
            F.count("TransactionID")           .alias("TotalTransactions"),
            F.sum("Quantity")                  .alias("TotalUnitsSold"),
            F.round(F.avg("TotalRevenue"),    2).alias("AvgOrderValue"),
            F.round(F.max("TotalRevenue"),    2).alias("MaxOrderValue"),
            F.countDistinct("StoreID")         .alias("ActiveStores"),
            F.countDistinct("ProductID")       .alias("ActiveProducts"),
            F.countDistinct("Date")            .alias("TradingDays"),
        )
        # Revenue per trading day — useful KPI
        .withColumn("RevenuePerTradingDay",
                    F.round(F.col("TotalRevenue") / F.col("TradingDays"), 2))
        .withColumn("_gold_loaded_at", F.current_timestamp())
        .withColumn("_gold_run_id",    F.lit(RUN_ID))
        .orderBy("Year", "Month")
    )

    rows = df_monthly.count()

    merge_gold_table(
        df         = df_monthly,
        table_path = GOLD_MONTHLY_PATH,
        table_name = GOLD_MONTHLY_NAME,
        merge_keys = ["Year", "Month"]
    )

    duration = round(time.time() - t_start, 2)
    write_audit(GOLD_MONTHLY_NAME, rows, "SUCCESS", duration)
    print(f"   {GOLD_MONTHLY_NAME} — {rows} rows in {duration}s")

except Exception as e:
    duration = round(time.time() - t_start, 2)
    write_audit(GOLD_MONTHLY_NAME, 0, "FAILED", duration, str(e))
    print(f"   {GOLD_MONTHLY_NAME} FAILED: {e}")



StatementMeta(, bea94974-b6f7-41e7-9c6b-ebf55260c66c, 10, Finished, Available, Finished, False)

  Created gold_monthly_summary
   gold_monthly_summary — 1 rows in 7.54s


In [9]:

# ── Notebook Cell 8 ──────────────────────────────────────────────────────────
# TITLE: Gold Table 4 — gold_store_product_performance
#
# One row per StoreID + ProductID + Year + Month.
# Answers: which product sells best in which store, by month?
# Includes RevenueRank — rank of each product within its store for that month.
# Power BI use: heatmap, matrix visual, top/bottom N product per store.

t_start = time.time()

try:
    from pyspark.sql.window import Window

    df_store_product = (df_silver
        .groupBy("StoreID", "ProductID", "Year", "Month")
        .agg(
            F.round(F.sum("TotalRevenue"),    2).alias("TotalRevenue"),
            F.sum("Quantity")                  .alias("UnitsSold"),
            F.count("TransactionID")           .alias("TransactionCount"),
            F.round(F.avg("UnitPrice"),        2).alias("AvgUnitPrice"),
            F.round(F.avg("TotalRevenue"),    2).alias("AvgOrderValue"),
        )
    )

    # Revenue rank of each product within its store for that month
    # Rank 1 = highest revenue product in that store that month
    store_month_window = Window.partitionBy("StoreID", "Year", "Month") \
                               .orderBy(F.desc("TotalRevenue"))

    df_store_product = (df_store_product
        .withColumn("RevenueRankInStore",
                    F.rank().over(store_month_window))
        .withColumn("_gold_loaded_at", F.current_timestamp())
        .withColumn("_gold_run_id",    F.lit(RUN_ID))
        .orderBy("StoreID", "Year", "Month", "RevenueRankInStore")
    )

    rows = df_store_product.count()

    merge_gold_table(
        df         = df_store_product,
        table_path = GOLD_STORE_PRODUCT_PATH,
        table_name = GOLD_STORE_PRODUCT_NAME,
        merge_keys = ["StoreID", "ProductID", "Year", "Month"]
    )

    duration = round(time.time() - t_start, 2)
    write_audit(GOLD_STORE_PRODUCT_NAME, rows, "SUCCESS", duration)
    print(f"   {GOLD_STORE_PRODUCT_NAME} — {rows} rows in {duration}s")

except Exception as e:
    duration = round(time.time() - t_start, 2)
    write_audit(GOLD_STORE_PRODUCT_NAME, 0, "FAILED", duration, str(e))
    print(f"   {GOLD_STORE_PRODUCT_NAME} FAILED: {e}")


StatementMeta(, bea94974-b6f7-41e7-9c6b-ebf55260c66c, 11, Finished, Available, Finished, False)

  Created gold_store_product_performance
   gold_store_product_performance — 8 rows in 7.44s


In [10]:


# ── Notebook Cell 9 ──────────────────────────────────────────────────────────
# TITLE: Run summary

total_duration = round(time.time() - RUN_START, 2)

print(f"\n{'═'*60}")
print("GOLD LAYER RUN COMPLETE")
print(f"Total duration : {total_duration}s")
print(f"{'═'*60}")

display(spark.sql(f"""
    SELECT
        table_name,
        gold_status,
        rows_written,
        duration_seconds,
        processed_at
    FROM {GOLD_AUDIT_NAME}
    WHERE run_id = '{RUN_ID}'
    ORDER BY processed_at
"""))



StatementMeta(, bea94974-b6f7-41e7-9c6b-ebf55260c66c, 12, Finished, Available, Finished, False)


════════════════════════════════════════════════════════════
GOLD LAYER RUN COMPLETE
Total duration : 86.46s
════════════════════════════════════════════════════════════


SynapseWidget(Synapse.DataFrame, 29b3f5c5-fb9e-4aab-ba72-2d5d3699c161)

In [11]:

# ── Notebook Cell 10 ──────────────────────────────────────────────────────────
# TITLE: Preview gold tables
# Optional — comment out in production


print("── gold_daily_store_revenue ──")
display(spark.table("gold_daily_store_revenue").orderBy("Date", "StoreID"))

print("── gold_daily_product_revenue ──")
display(spark.table("gold_daily_product_revenue").orderBy("Date", "ProductID"))

print("── gold_monthly_summary ──")
display(spark.table("gold_monthly_summary").orderBy("Year", "Month"))

print("── gold_store_product_performance ──")
display(spark.table("gold_store_product_performance").orderBy("StoreID", "Year", "Month", "RevenueRankInStore"))

StatementMeta(, bea94974-b6f7-41e7-9c6b-ebf55260c66c, 13, Finished, Available, Finished, False)

── gold_daily_store_revenue ──


SynapseWidget(Synapse.DataFrame, 7dace68a-2f64-4cd7-a356-e3072d0a8cc2)

── gold_daily_product_revenue ──


SynapseWidget(Synapse.DataFrame, c550380a-edf0-4f37-bbcb-8157dfb1340f)

── gold_monthly_summary ──


SynapseWidget(Synapse.DataFrame, e04caf1c-b2e8-476f-942a-5ec99c593fcb)

── gold_store_product_performance ──


SynapseWidget(Synapse.DataFrame, 33a9e36b-6f2b-48b7-a1ec-11ce6ecfeec4)